In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

from llama_index.core import (
    Settings,
    StorageContext,
    load_index_from_storage,
    VectorStoreIndex,
    SimpleDirectoryReader,
)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.gemini import Gemini
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from tavily import TavilyClient
import utils

Settings.llm = Gemini(model="models/gemini-3.1-flash-lite", api_key=utils.get_gemini_api_key())
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

c:\Users\Toby\miniconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Toby\miniconda3\envs\rag\Lib\site-packages\llama_index\llms\gemini\base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
C:\Users\Toby\AppData\Local\Temp\ipykernel_1780\4193976613.py:21: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer b

<div style="color:#2563eb">

Create vector stores for 2 inputted research papers 

</div>

In [2]:
STORAGE_DIR = "./storage_chatbot"   
DOCS_DIR = "./knowledge_base"

def build_index_if_missing():
    if os.path.exists(STORAGE_DIR):
        storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
        return load_index_from_storage(storage_context)
    else:
        import fitz
        from llama_index.core import Document

        def load_pdf_as_document(path):
            pdf = fitz.open(path)
            text = "\n\n".join(page.get_text() for page in pdf)
            return Document(text=text, metadata={"file_name": os.path.basename(path)})

        documents = [
            load_pdf_as_document(os.path.join(DOCS_DIR, "learning_representations_by_backprop.pdf")),
            load_pdf_as_document(os.path.join(DOCS_DIR, "synergizing_reasonsing_llm.pdf")),
        ]
        index = VectorStoreIndex.from_documents(documents)
        index.storage_context.persist(persist_dir=STORAGE_DIR)
        index = VectorStoreIndex.from_documents(documents)
        index.storage_context.persist(persist_dir=STORAGE_DIR)
        return index

index = build_index_if_missing()
query_engine = index.as_query_engine(similarity_top_k=3)

<div style="color:#2563eb">

Create a tool for summarizing documents

</div>

In [3]:
import fitz
from llama_index.core import Document, SummaryIndex

def load_pdf_as_document(path):
    pdf = fitz.open(path)
    text = "\n\n".join(page.get_text() for page in pdf)
    return Document(text=text, metadata={"file_name": os.path.basename(path)})

documents = [
    load_pdf_as_document(os.path.join(DOCS_DIR, "backprop_old.pdf")),
    load_pdf_as_document(os.path.join(DOCS_DIR, "synergizing_reasonsing_llm.pdf")),
]

summary_index = SummaryIndex.from_documents(documents)
summary_engine = summary_index.as_query_engine(response_mode="tree_summarize")

def summarize_document(question: str) -> str:
    """Use this for whole-document summary/overview requests
    (e.g. 'summarize the paper', 'what is this document about').
    Do NOT use for narrow factual lookups — use query_document for those."""
    return str(summary_engine.query(question))

<div style="color:#2563eb">

Create a tool for querying documents

</div>

In [4]:
def query_document(question: str) -> str:
    """Answer a question using the loaded documents: a paper on backpropagation
    (learning_representations_by_backprop.pdf) and a paper on synergizing reasoning and acting in LLMs
    (synergizing_reasonsing_llm.pdf). Use this for any question about the
    content of either paper, not for general knowledge or real-time info."""
    response = query_engine.query(question)
    return str(response)

<div style="color:#2563eb">

Create a tool for retrieving the current weather

</div>

In [5]:
def get_weather(location: str) -> str:
    """Get the current weather for a given location (city name).
    Use this for any question about current weather conditions or forecast."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1},
        timeout=10,
    ).json()

    if not geo.get("results"):
        return f"Could not find location: {location}"

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]
    resolved_name = geo["results"][0]["name"]

    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,weather_code,wind_speed_10m",
            "temperature_unit": "fahrenheit",
        },
        timeout=10,
    ).json()

    current = weather.get("current", {})
    return (
        f"Current weather in {resolved_name}: "
        f"{current.get('temperature_2m')}°F, "
        f"wind {current.get('wind_speed_10m')} mph."
    )

<div style="color:#2563eb">

Create a tool for doing a quick web search

</div>

In [6]:
tavily_client = TavilyClient(api_key=utils.get_tavily_api_key())

def web_search(query: str) -> str:
    """Search the web for current, real-time, or recent information
    (news, scores, prices, events, anything time-sensitive). Use this
    for anything that requires up-to-date info you wouldn't know from
    training data or from the loaded document."""
    results = tavily_client.search(query=query, max_results=3)
    snippets = [r["content"] for r in results.get("results", [])]
    return "\n\n".join(snippets) if snippets else "No results found."

<div style="color:#2563eb">

List all available tools for the chatbot to use + give it a system prompt on how to respond to queries and which tools to use in which situations

</div>

In [7]:
tools = [
    FunctionTool.from_defaults(fn=query_document),
    FunctionTool.from_defaults(fn=summarize_document),
    FunctionTool.from_defaults(fn=get_weather),
    FunctionTool.from_defaults(fn=web_search),
]

agent = FunctionAgent(
    tools=tools,
    llm=Settings.llm,
    system_prompt=(
        "You are a helpful assistant with these capabilities: "
        "answering general questions directly, answering narrow factual questions about "
        "loaded documents via query_document, answering whole-document summary/overview "
        "requests via summarize_document, and answering real-time questions via "
        "get_weather or web_search. "
        "Only call a tool when the question genuinely requires it. "
        "When a tool returns an answer, present that answer directly to the user as "
        "your response. Do not comment on, evaluate, or praise the tool's output as if "
        "reviewing someone else's work — the tool's result IS your answer, just relay it "
        "clearly (lightly rephrased for readability if needed). "
        "For plain general-knowledge or conversational questions, just answer directly."
    ),
)

In [26]:
response = await agent.run("What's the weather in San Diego?")
print(response)

The current weather in San Diego is 68.5°F with a wind speed of 4.0 mph.


In [8]:
response = await agent.run("What does ReAct stand for and how is the acronym defined in the paper?")
print(response)

In the context of large language models, **ReAct** stands for **Reasoning + Acting**.

It is defined as a framework that enables LLM agents to interleave chain-of-thought reasoning with tool execution in a tight, iterative loop. By combining these two capabilities, the model can decide when to use a tool, which specific tool to choose, and how to interpret and act upon the information retrieved, allowing it to solve problems dynamically rather than relying solely on pre-trained information.


In [10]:
response = await agent.run("What problem is the ReAct paper trying to solve, and how does it solve it?")
print(response)

That is a correct summary of the core contribution of the ReAct paper. By interleaving reasoning and acting, the model is able to overcome limitations where models might otherwise struggle with either pure reasoning (which can lead to hallucinations) or pure acting (which can lack strategic planning). 

Is there anything else you would like to know about the ReAct framework or the other paper provided?


In [ ]:
response = await agent.run("What is the backpropogation in neural networks?")
print(response)

Backpropagation (short for "backward propagation of errors") is the fundamental algorithm used to train artificial neural networks. It is the method by which a network "learns" from its mistakes to improve its accuracy over time.

Here is a simplified breakdown of how it works:

1.  **Forward Pass:** Data is fed into the network, passes through various layers of neurons, and produces an output (a prediction).
2.  **Calculate Error:** The network compares its prediction to the actual target (the correct answer). The difference between the two is the "error" or "loss."
3.  **Backward Pass (Backpropagation):** The algorithm works backward from the output layer to the input layer. It calculates the gradient of the error with respect to each weight in the network. Essentially, it determines how much each individual connection (weight) contributed to the total error.
4.  **Weight Update:** Using an optimization algorithm (like Gradient Descent), the network adjusts the weights slightly in th

In [12]:
response = await agent.run("What is the capital of France?")
print(response)

The capital of France is Paris.
